## Kütüphaneleri Import Et

In [1]:
import pandas as pd
import html
import unicodedata
import numpy as np
import re
import ast
import pyarrow

## Veriyi Oku

In [2]:
data_path = "../data/original/recipes.csv"
data = pd.read_csv(data_path)

In [3]:
df = data.copy()

In [4]:
df.head(3)

,RecipeId,Name,AuthorId,AuthorName,CookTime,PrepTime,TotalTime,DatePublished,Description,Images,...,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeServings,RecipeYield,RecipeInstructions
0,38,Low-Fat Berry Blue Frozen Dessert,1533,Dancer,PT24H,PT45M,PT24H45M,1999-08-09T21:46:00Z,Make and share this Low-Fat Berry Blue Frozen ...,"c(""https://img.sndimg.com/food/image/upload/w_...",...,1.3,8.0,29.8,37.1,3.6,30.2,3.2,4.0,NaN,"c(""Toss 2 cups berries with sugar."", ""Let stan..."
1,39,Biryani,1567,elly9812,PT25M,PT4H,PT4H25M,1999-08-29T13:12:00Z,Make and share this Biryani recipe from Food.com.,"c(""https://img.sndimg.com/food/image/upload/w_...",...,16.6,372.8,368.4,84.4,9.0,20.4,63.4,6.0,NaN,"c(""Soak saffron in warm milk for 5 minutes and..."
2,40,Best Lemonade,1566,Stephen Little,PT5M,PT30M,PT35M,1999-09-05T19:52:00Z,This is from one of my first Good House Keepi...,"c(""https://img.sndimg.com/food/image/upload/w_...",...,0.0,0.0,1.8,81.5,0.4,77.2,0.3,4.0,NaN,"c(""Into a 1 quart Jar with tight fitting lid, ..."


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 522517 entries, 0 to 522516
Data columns (total 28 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   RecipeId                    522517 non-null  int64  
 1   Name                        522517 non-null  object 
 2   AuthorId                    522517 non-null  int64  
 3   AuthorName                  522517 non-null  object 
 4   CookTime                    439972 non-null  object 
 5   PrepTime                    522517 non-null  object 
 6   TotalTime                   522517 non-null  object 
 7   DatePublished               522517 non-null  object 
 8   Description                 522512 non-null  object 
 9   Images                      522516 non-null  object 
 10  RecipeCategory              521766 non-null  object 
 11  Keywords                    505280 non-null  object 
 12  RecipeIngredientQuantities  522514 non-null  object 
 13  RecipeIngredie

## Kullanılmayacak Sütunları Sil

In [6]:
deleted_columns = [
    "AuthorId",
    "AuthorName",
    "CookTime",
    "PrepTime",
    "DatePublished",
    "Images",
    "ReviewCount",
    "SaturatedFatContent",
    "CholesterolContent",
    "SodiumContent",
    "FiberContent",
    "RecipeServings",
    "RecipeYield",
]

featured_df = df.drop(columns=deleted_columns)
featured_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 522517 entries, 0 to 522516
Data columns (total 15 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   RecipeId                    522517 non-null  int64  
 1   Name                        522517 non-null  object 
 2   TotalTime                   522517 non-null  object 
 3   Description                 522512 non-null  object 
 4   RecipeCategory              521766 non-null  object 
 5   Keywords                    505280 non-null  object 
 6   RecipeIngredientQuantities  522514 non-null  object 
 7   RecipeIngredientParts       522517 non-null  object 
 8   AggregatedRating            269294 non-null  float64
 9   Calories                    522517 non-null  float64
 10  FatContent                  522517 non-null  float64
 11  CarbohydrateContent         522517 non-null  float64
 12  SugarContent                522517 non-null  float64
 13  ProteinContent

## PascalCase -> snake_case

In [7]:
# CamelCase olan isimleri snake_case'e çevirme
def camel_to_snake(name: str) -> str:
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    return re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower()
featured_df.columns = [camel_to_snake(col) for col in featured_df.columns]

In [8]:
featured_df.columns

Index(['recipe_id', 'name', 'total_time', 'description', 'recipe_category',
       'keywords', 'recipe_ingredient_quantities', 'recipe_ingredient_parts',
       'aggregated_rating', 'calories', 'fat_content', 'carbohydrate_content',
       'sugar_content', 'protein_content', 'recipe_instructions'],
      dtype='object')

## Eksik Veri Manipülasyonu ve Silme İşlemleri

### Eksik Verilerin Tespiti
Bu uygulama için mevcut veri setinin (yaklaşık 500 bin veri var) küçük bir kısmı kullanılacağı için eksik veriler silinecektir.

In [9]:
featured_df.isnull().sum().sort_values(ascending=False)

aggregated_rating               253223
keywords                         17237
recipe_category                    751
description                          5
recipe_ingredient_quantities         3
recipe_id                            0
name                                 0
total_time                           0
recipe_ingredient_parts              0
calories                             0
fat_content                          0
carbohydrate_content                 0
sugar_content                        0
protein_content                      0
recipe_instructions                  0
dtype: int64

In [10]:
missing_frac = featured_df.isnull().mean().sort_values(ascending=False)
missing_frac

aggregated_rating               0.484622
keywords                        0.032988
recipe_category                 0.001437
description                     0.000010
recipe_ingredient_quantities    0.000006
recipe_id                       0.000000
name                            0.000000
total_time                      0.000000
recipe_ingredient_parts         0.000000
calories                        0.000000
fat_content                     0.000000
carbohydrate_content            0.000000
sugar_content                   0.000000
protein_content                 0.000000
recipe_instructions             0.000000
dtype: float64

In [11]:
# calories 0 olanları none yapma
featured_df.loc[featured_df['calories'] == 0, 'calories'] = None

In [12]:
# Eksik verilerin olduğu satırları silme
missing_data_columns = [
    "aggregated_rating",
    "keywords",
    "recipe_category",
    "description",
    "recipe_ingredient_quantities",
    "calories",
]
featured_df.dropna(subset=missing_data_columns, inplace=True)
featured_df.reset_index(drop=True, inplace=True)


In [13]:
featured_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 261636 entries, 0 to 261635
Data columns (total 15 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   recipe_id                     261636 non-null  int64  
 1   name                          261636 non-null  object 
 2   total_time                    261636 non-null  object 
 3   description                   261636 non-null  object 
 4   recipe_category               261636 non-null  object 
 5   keywords                      261636 non-null  object 
 6   recipe_ingredient_quantities  261636 non-null  object 
 7   recipe_ingredient_parts       261636 non-null  object 
 8   aggregated_rating             261636 non-null  float64
 9   calories                      261636 non-null  float64
 10  fat_content                   261636 non-null  float64
 11  carbohydrate_content          261636 non-null  float64
 12  sugar_content                 261636 non-nul

In [14]:
featured_df.head(3)

,recipe_id,name,total_time,description,recipe_category,keywords,recipe_ingredient_quantities,recipe_ingredient_parts,aggregated_rating,calories,fat_content,carbohydrate_content,sugar_content,protein_content,recipe_instructions
0,38,Low-Fat Berry Blue Frozen Dessert,PT24H45M,Make and share this Low-Fat Berry Blue Frozen ...,Frozen Desserts,"c(""Dessert"", ""Low Protein"", ""Low Cholesterol"",...","c(""4"", ""1/4"", ""1"", ""1"")","c(""blueberries"", ""granulated sugar"", ""vanilla ...",4.5,170.9,2.5,37.1,30.2,3.2,"c(""Toss 2 cups berries with sugar."", ""Let stan..."
1,39,Biryani,PT4H25M,Make and share this Biryani recipe from Food.com.,Chicken Breast,"c(""Chicken Thigh & Leg"", ""Chicken"", ""Poultry"",...","c(""1"", ""4"", ""2"", ""2"", ""8"", ""1/4"", ""8"", ""1/2"", ...","c(""saffron"", ""milk"", ""hot green chili peppers""...",3.0,1110.7,58.8,84.4,20.4,63.4,"c(""Soak saffron in warm milk for 5 minutes and..."
2,40,Best Lemonade,PT35M,This is from one of my first Good House Keepi...,Beverages,"c(""Low Protein"", ""Low Cholesterol"", ""Healthy"",...","c(""1 1/2"", ""1"", NA, ""1 1/2"", NA, ""3/4"")","c(""sugar"", ""lemons, rind of"", ""lemon, zest of""...",4.5,311.1,0.2,81.5,77.2,0.3,"c(""Into a 1 quart Jar with tight fitting lid, ..."


## Özellik Mühendisliği - Yeni Özellikler Ekleme

### Object (R Vector) -> Python List
recipe_instructions, recipe_ingredient_parts, keywords, recipe_ingredient_quantities sütunları bir R c(...) vektörünü andıran stringler içeriyor. Bunları python listelerine parse eden fonksiyonlar yazalım.

In [15]:
def parse_r_vector(r_vector_str: str) -> list:
    if pd.isna(r_vector_str) or r_vector_str.strip() == "":
        return []
    try:
        python_list = ast.literal_eval(r_vector_str.replace('c(', '[').replace(')', ']').replace('"', "'"))
        if isinstance(python_list, list):
            return [str(item).strip() for item in python_list]
        else:
            return []
    except (ValueError, SyntaxError):
        return []

In [16]:
featured_df['recipe_instructions'] = featured_df['recipe_instructions'].apply(parse_r_vector)
featured_df['recipe_ingredient_parts'] = featured_df['recipe_ingredient_parts'].apply(parse_r_vector)
featured_df['keywords'] = featured_df['keywords'].apply(parse_r_vector)
featured_df['recipe_ingredient_quantities'] = featured_df['recipe_ingredient_quantities'].apply(parse_r_vector)

<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal
<unknown>:2: SyntaxWarning: invalid decimal literal


In [17]:
featured_df.head(3)

,recipe_id,name,total_time,description,recipe_category,keywords,recipe_ingredient_quantities,recipe_ingredient_parts,aggregated_rating,calories,fat_content,carbohydrate_content,sugar_content,protein_content,recipe_instructions
0,38,Low-Fat Berry Blue Frozen Dessert,PT24H45M,Make and share this Low-Fat Berry Blue Frozen ...,Frozen Desserts,"[Dessert, Low Protein, Low Cholesterol, Health...","[4, 1/4, 1, 1]","[blueberries, granulated sugar, vanilla yogurt...",4.5,170.9,2.5,37.1,30.2,3.2,[]
1,39,Biryani,PT4H25M,Make and share this Biryani recipe from Food.com.,Chicken Breast,"[Chicken Thigh & Leg, Chicken, Poultry, Meat, ...",[],"[saffron, milk, hot green chili peppers, onion...",3.0,1110.7,58.8,84.4,20.4,63.4,[Soak saffron in warm milk for 5 minutes and p...
2,40,Best Lemonade,PT35M,This is from one of my first Good House Keepi...,Beverages,"[Low Protein, Low Cholesterol, Healthy, Summer...",[],"[sugar, lemons, rind of, lemon, zest of, fresh...",4.5,311.1,0.2,81.5,77.2,0.3,"[Into a 1 quart Jar with tight fitting lid, pu..."


In [18]:
# featured_df'in recipe_instructions, recipe_ingredient_parts, keywords, recipe_ingredient_quantities sütunlarındaki boş liste sayısına bakalım.
empty_list_counts = {
    'recipe_instructions': featured_df['recipe_instructions'].apply(lambda x: len(x) == 0).sum(),
    'recipe_ingredient_parts': featured_df['recipe_ingredient_parts'].apply(lambda x: len(x) == 0).sum(),
    'keywords': featured_df['keywords'].apply(lambda x: len(x) == 0).sum(),
    'recipe_ingredient_quantities': featured_df['recipe_ingredient_quantities'].apply(lambda x: len(x) == 0).sum(),
}
empty_list_counts

{'recipe_instructions': 37769,
 'recipe_ingredient_parts': 9237,
 'keywords': 17521,
 'recipe_ingredient_quantities': 90377}

In [19]:
# featured_df'in recipe_instructions, recipe_ingredient_parts, keywords, recipe_ingredient_quantities sütunlarındaki boş liste olan satırları silelim
for column in ['recipe_instructions', 'recipe_ingredient_parts', 'keywords', 'recipe_ingredient_quantities']:
    featured_df = featured_df[featured_df[column].apply(lambda x: len(x) > 0)]
featured_df.reset_index(drop=True, inplace=True)

In [20]:
featured_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132662 entries, 0 to 132661
Data columns (total 15 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   recipe_id                     132662 non-null  int64  
 1   name                          132662 non-null  object 
 2   total_time                    132662 non-null  object 
 3   description                   132662 non-null  object 
 4   recipe_category               132662 non-null  object 
 5   keywords                      132662 non-null  object 
 6   recipe_ingredient_quantities  132662 non-null  object 
 7   recipe_ingredient_parts       132662 non-null  object 
 8   aggregated_rating             132662 non-null  float64
 9   calories                      132662 non-null  float64
 10  fat_content                   132662 non-null  float64
 11  carbohydrate_content          132662 non-null  float64
 12  sugar_content                 132662 non-nul

In [21]:
featured_df.head(3)

,recipe_id,name,total_time,description,recipe_category,keywords,recipe_ingredient_quantities,recipe_ingredient_parts,aggregated_rating,calories,fat_content,carbohydrate_content,sugar_content,protein_content,recipe_instructions
0,41,Carina's Tofu-Vegetable Kebabs,PT24H20M,This dish is best prepared a day in advance to...,Soy/Tofu,"[Beans, Vegetable, Low Cholesterol, Weeknight,...","[12, 1, 2, 1, 10, 1, 3, 2, 2, 2, 1, 2, 1/2, 1/...","[extra firm tofu, eggplant, zucchini, mushroom...",4.5,536.1,24.0,64.2,32.1,29.3,"[Drain the tofu, carefully squeezing out exces..."
1,42,Cabbage Soup,PT50M,Make and share this Cabbage Soup recipe from F...,Vegetable,"[Low Protein, Vegan, Low Cholesterol, Healthy,...","[46, 4, 1, 2, 1]","[plain tomato juice, cabbage, onion, carrots, ...",4.5,103.6,0.4,25.1,17.7,4.3,"[Mix everything together and bring to a boil.,..."
2,45,Buttermilk Pie With Gingersnap Crumb Crust,PT1H20M,Make and share this Buttermilk Pie With Ginger...,Pie,"[Dessert, Healthy, Weeknight, Oven, < 4 Hours]","[3/4, 1, 1, 2, 3, 1/4, 1, 1/2, 1/2, 2]","[sugar, margarine, egg, flour, salt, buttermil...",4.0,228.0,7.1,37.5,24.7,4.2,"[Preheat oven to 350°F., Make pie crust, using..."


### n_ingredients
Tarif içinde kullanılan ürün sayısı

In [22]:
featured_df['n_ingredients'] = featured_df['recipe_ingredient_parts'].apply(lambda x: len(x) if isinstance(x, (list, tuple)) else 0)

In [23]:
featured_df.head(3)

,recipe_id,name,total_time,description,recipe_category,keywords,recipe_ingredient_quantities,recipe_ingredient_parts,aggregated_rating,calories,fat_content,carbohydrate_content,sugar_content,protein_content,recipe_instructions,n_ingredients
0,41,Carina's Tofu-Vegetable Kebabs,PT24H20M,This dish is best prepared a day in advance to...,Soy/Tofu,"[Beans, Vegetable, Low Cholesterol, Weeknight,...","[12, 1, 2, 1, 10, 1, 3, 2, 2, 2, 1, 2, 1/2, 1/...","[extra firm tofu, eggplant, zucchini, mushroom...",4.5,536.1,24.0,64.2,32.1,29.3,"[Drain the tofu, carefully squeezing out exces...",14
1,42,Cabbage Soup,PT50M,Make and share this Cabbage Soup recipe from F...,Vegetable,"[Low Protein, Vegan, Low Cholesterol, Healthy,...","[46, 4, 1, 2, 1]","[plain tomato juice, cabbage, onion, carrots, ...",4.5,103.6,0.4,25.1,17.7,4.3,"[Mix everything together and bring to a boil.,...",5
2,45,Buttermilk Pie With Gingersnap Crumb Crust,PT1H20M,Make and share this Buttermilk Pie With Ginger...,Pie,"[Dessert, Healthy, Weeknight, Oven, < 4 Hours]","[3/4, 1, 1, 2, 3, 1/4, 1, 1/2, 1/2, 2]","[sugar, margarine, egg, flour, salt, buttermil...",4.0,228.0,7.1,37.5,24.7,4.2,"[Preheat oven to 350°F., Make pie crust, using...",8


### ingredients
Tarifte kullanılan malzeme ve miktarları recipe_ingredient_parts ve recipe_ingredient_quantities sütunlarında tutuluyor. Bu listeleri birleştirerek tek bir liste oluşturalım.

In [24]:
def combine_ingredients_and_quantities(ingredients, quantities):
    if not isinstance(ingredients, list) or not isinstance(quantities, list):
        return []
    combined = []
    for ingredient, quantity in zip(ingredients, quantities):
        combined.append(f"{quantity} {ingredient}".strip())
    return combined

In [25]:
featured_df['ingredients'] = featured_df.apply(
    lambda row: combine_ingredients_and_quantities(
        row['recipe_ingredient_parts'], row['recipe_ingredient_quantities']
    ), axis=1
)
featured_df.drop(columns=["recipe_ingredient_quantities"], inplace=True)

In [26]:
featured_df.head(3)

,recipe_id,name,total_time,description,recipe_category,keywords,recipe_ingredient_parts,aggregated_rating,calories,fat_content,carbohydrate_content,sugar_content,protein_content,recipe_instructions,n_ingredients,ingredients
0,41,Carina's Tofu-Vegetable Kebabs,PT24H20M,This dish is best prepared a day in advance to...,Soy/Tofu,"[Beans, Vegetable, Low Cholesterol, Weeknight,...","[extra firm tofu, eggplant, zucchini, mushroom...",4.5,536.1,24.0,64.2,32.1,29.3,"[Drain the tofu, carefully squeezing out exces...",14,"[12 extra firm tofu, 1 eggplant, 2 zucchini, 1..."
1,42,Cabbage Soup,PT50M,Make and share this Cabbage Soup recipe from F...,Vegetable,"[Low Protein, Vegan, Low Cholesterol, Healthy,...","[plain tomato juice, cabbage, onion, carrots, ...",4.5,103.6,0.4,25.1,17.7,4.3,"[Mix everything together and bring to a boil.,...",5,"[46 plain tomato juice, 4 cabbage, 1 onion, 2 ..."
2,45,Buttermilk Pie With Gingersnap Crumb Crust,PT1H20M,Make and share this Buttermilk Pie With Ginger...,Pie,"[Dessert, Healthy, Weeknight, Oven, < 4 Hours]","[sugar, margarine, egg, flour, salt, buttermil...",4.0,228.0,7.1,37.5,24.7,4.2,"[Preheat oven to 350°F., Make pie crust, using...",8,"[3/4 sugar, 1 margarine, 1 egg, 2 flour, 3 sal..."


### Hour To Minutes
total_time sütunu string formatta saat ve dakika verilerini barındırıyor. String'lerden zaman bilgisini çıkarıp saat -> dakika dönüşümü yapalım.

Örn:
"PT1H30M" -> 1 * 60 + 30, "PT1H" -> 60, "PT30M" -> 30.

In [27]:
def parse_iso_duration(duration_str):
    if pd.isna(duration_str):
        return None
    hours = 0
    minutes = 0
    hour_match = re.search(r'PT(\d+)H', duration_str)
    minute_match = re.search(r'PT(?:\d+H)?(\d+)M', duration_str)
    if hour_match:
        hours = int(hour_match.group(1))
    if minute_match:
        minutes = int(minute_match.group(1))
    return hours * 60 + minutes

In [28]:
featured_df['total_time_minutes'] = featured_df['total_time'].apply(parse_iso_duration)
featured_df.drop(columns=['total_time'], inplace=True)

### Besin Değeri Özetleri
Protein, şeker, yağ ve karbonhidrat özellikleri için yüzdelik dilimlerini ifade eden ve grafikselleştirilebilir değer çıkarımları yapalım. Toplam besin değeri 0 olan değerleri silelim.

In [29]:
nutri_cols = ['protein_content','fat_content','carbohydrate_content','sugar_content']
total = featured_df[nutri_cols].sum(axis=1)

for c in nutri_cols:
    featured_df[f'{c}_perc'] = np.where(total > 0, (featured_df[c] / total) * 100.0, np.nan)

perc_cols = [f'{c}_perc' for c in nutri_cols]
featured_df[perc_cols] = featured_df[perc_cols].round(2)

mask = total > 0
print('Silinen satır sayısı (total == 0):', (~mask).sum())
featured_df = featured_df.loc[mask].reset_index(drop=True)


Silinen satır sayısı (total == 0): 102


### recipe_id
tariflerin id değerlerini ifade eden bu sütunun değerlerini 0'dan başlayacak şekilde sıralı bir şekilde yeniden yazalım

In [30]:
featured_df = featured_df.reset_index(drop=True)
featured_df['recipe_id'] = featured_df.index.astype(int)

In [31]:
featured_df["recipe_id"].min(), featured_df["recipe_id"].max()

(0, 132559)

### Tür Dönüşümü
Verilerin gereksiz yer kaplamaması için kendilerine uygun tür tanımları yapalım.

In [32]:
int32_cols = ['recipe_id', 'n_ingredients', 'total_time_minutes']
float32_cols = ['aggregated_rating', 'calories'] + perc_cols + nutri_cols
featured_df[int32_cols] = featured_df[int32_cols].astype(np.int32)
featured_df[float32_cols] = featured_df[float32_cols].astype(np.float32)

### Embedding Veri Hazırlığı
Embedding işlemi için kullanılacak verileri toplu bir metin haline getirip metin önişleme (harflerin küçük yazılması, boşlukların kaldırılması vs.) yapalım.

In [33]:
featured_df["embedded_text"] = (
    "name: " + featured_df["name"].astype(str) + ". " +
    "description: " + featured_df["description"].astype(str) + ". " +
    "keywords: " + featured_df["keywords"].astype(str) + ". " +
    "category: " + featured_df["recipe_category"].astype(str) + ". " +
    "ingredient parts: " + featured_df["recipe_ingredient_parts"].astype(str) + ". " +
    "instructions: " + featured_df["recipe_instructions"].astype(str) + ". "
)

In [34]:
featured_df['embedded_text'] = (
    featured_df['embedded_text']
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

## Veri Son Kontrol

In [35]:
featured_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132560 entries, 0 to 132559
Data columns (total 21 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   recipe_id                  132560 non-null  int32  
 1   name                       132560 non-null  object 
 2   description                132560 non-null  object 
 3   recipe_category            132560 non-null  object 
 4   keywords                   132560 non-null  object 
 5   recipe_ingredient_parts    132560 non-null  object 
 6   aggregated_rating          132560 non-null  float32
 7   calories                   132560 non-null  float32
 8   fat_content                132560 non-null  float32
 9   carbohydrate_content       132560 non-null  float32
 10  sugar_content              132560 non-null  float32
 11  protein_content            132560 non-null  float32
 12  recipe_instructions        132560 non-null  object 
 13  n_ingredients              13

In [36]:
featured_df.head(3)

,recipe_id,name,description,recipe_category,keywords,recipe_ingredient_parts,aggregated_rating,calories,fat_content,carbohydrate_content,...,protein_content,recipe_instructions,n_ingredients,ingredients,total_time_minutes,protein_content_perc,fat_content_perc,carbohydrate_content_perc,sugar_content_perc,embedded_text
0,0,Carina's Tofu-Vegetable Kebabs,This dish is best prepared a day in advance to...,Soy/Tofu,"[Beans, Vegetable, Low Cholesterol, Weeknight,...","[extra firm tofu, eggplant, zucchini, mushroom...",4.5,536.099976,24.0,64.199997,...,29.299999,"[Drain the tofu, carefully squeezing out exces...",14,"[12 extra firm tofu, 1 eggplant, 2 zucchini, 1...",1460,19.59,16.040001,42.91,21.459999,name: carina's tofu-vegetable kebabs. descript...
1,1,Cabbage Soup,Make and share this Cabbage Soup recipe from F...,Vegetable,"[Low Protein, Vegan, Low Cholesterol, Healthy,...","[plain tomato juice, cabbage, onion, carrots, ...",4.5,103.599998,0.4,25.100000,...,4.300000,"[Mix everything together and bring to a boil.,...",5,"[46 plain tomato juice, 4 cabbage, 1 onion, 2 ...",50,9.05,0.840000,52.84,37.259998,name: cabbage soup. description: make and shar...
2,2,Buttermilk Pie With Gingersnap Crumb Crust,Make and share this Buttermilk Pie With Ginger...,Pie,"[Dessert, Healthy, Weeknight, Oven, < 4 Hours]","[sugar, margarine, egg, flour, salt, buttermil...",4.0,228.000000,7.1,37.500000,...,4.200000,"[Preheat oven to 350°F., Make pie crust, using...",8,"[3/4 sugar, 1 margarine, 1 egg, 2 flour, 3 sal...",80,5.71,9.660000,51.02,33.610001,name: buttermilk pie with gingersnap crumb cru...


In [37]:
final_df = featured_df.sample(10000, random_state=42)

## Veri Kaydetme
Vektörizasyon için hazırlanmış ve uygulamada kullanılacak verileri kaydedelim.

In [38]:
vector_df = final_df[["recipe_id", "embedded_text"]].copy()
vector_df.to_parquet('../data/processed/vector_metadata.parquet', index=False, engine="pyarrow", compression="snappy")

In [39]:
master_df = final_df.drop(columns=['recipe_ingredient_parts', 'embedded_text'], axis=1)
master_df.to_parquet('../data/processed/master.parquet', index=False, engine="pyarrow", compression="snappy")